# nb8 — Hướng A: chuẩn hóa Telex/VNI đầu vào + A/B frozen Run 2 (eval-only · Kaggle T4)

Thực thi `REPORT.md` §15.1 qua plan `.kilo/plans/1790300265078-telex-normalize-mlm-oracle.md` (quyết định 25/09): WA telex_real chỉ ~15% trên eval-real (mục 12.2) → thử sửa **input trước model**: detector + bảng đảo chuyển tự tất định (telex/VNI) → adapter Run 2 frozen → post-processing frozen G3-r1r2-r3k1 (nb6) → so A/B với pipeline hiện có.

**Thiết kế normalizer (tất định, gold-agnostic — plan §nb8)**: token eligible nếu (a) `(lower) ∉ SYLL_SET` — từ đã hợp lệ không đụng, kể cả khi có expansion; (b) thuần ASCII/`đ`, không mang dấu tổ hợp; (c) bảng đảo chuyển tự (dựng từ `telex_candidates` nb7 × **bảng âm tiết 7.884 nguồn công khai độc lập**) cho ra **đúng 1** expansion (2 style telex+VNI đã dedupe trước khi đếm; 0 hoặc ≥2 → bỏ vị trí, bảo thủ). Guard ngữ cảnh: chỉ áp nếu câu chứa **≥3 token có dấu/không-ASCII** (tránh câu tiếng Anh thuần, URL).

**Chỉ chạy 1 nhánh inference mới (nhánh normalized)**; nhánh raw dùng lại predictions có sẵn (nb3 cho val/test, nb5 cho eval-real) — tránh so 2 run GPU nondeterministic.

**Ngưỡng verdict PRE-REGISTERED** (chốt TRƯỚC khi nhìn kết quả, in ở §0):
- **PASS Hướng A** nếu cả 3: (i) ΔWA telex_real (norm+PP − raw+PP) **≥ +5pp**; (ii) Δover-correction test ≤ **+0,1pp**; (iii) ΔF1 test (norm+PP − raw+PP) ≥ **−0,1pp** (không tệ đi đáng kể toàn cục).
- **E3 guard ablation**: câu sạch test bị normalizer đụng phải **= 0** — không đạt → điều tra trước khi verdict, không âm thầm nới ngưỡng.
- Slice telex test < 100 câu → verdict đọc kèm eval-real + ghi hạn chế. FAIL → negative result vào REPORT (telex là giới hạn khó chạm bằng cả 2 đường augmentation lẫn normalization).

**Input**: nb3 (`predictions_{val,test}_run2.jsonl`, `lora_adapter_run2/`) · nb6 (`postprocess_config.json` — nạp frozen, KHÔNG re-tune) · nb2 (`syllable_table.json`) · nb1 (`test_aligned.jsonl` đối chiếu) · eval-real: file `predictions_evalreal_run2.jsonl` của nb5 (không có → load HF `nrl-ai/vn-spell-correction-eval-real`, CC0, cần Internet ON; khi đó nhánh raw eval-real chạy lại trong phiên — deviation, ghi report).

**Output**: `telex_normalizer_rules.json` · `predictions_{val,test}_run2_norm.jsonl` · `predictions_evalreal_run2_norm.jsonl` · `predictions_*_run2_norm_postproc.jsonl` · `telex_ab_report.json`.

**Checklist chống leakage (`DESIGN.md` §9)**: normalizer chỉ từ bảng chuyển tự công khai + bảng âm tiết độc lập (không nhìn nhãn) · PP nạp frozen nb6 (tune trên val Run 2) · eval-real không tham gia tune gì · seed 42.

## Nội dung
0. Cấu hình + guard môi trường + ngưỡng pre-registered + tự dò input
1. Cell hàm dùng chung (copy nguyên vẹn từ nb5 — `align-v1`) + `evaluate_predictions` + helper word acc (nb5)
2. Bảng đảo chuyển tự + normalizer + sanity nhân tạo
3. Dữ liệu + thống kê tác động + E3 guard ablation
4. Inference nhánh chuẩn hóa (T4 · greedy)
5. Post-processing frozen G3 + bảng A/B
6. Verdict pre-registered + QA + xuất file

In [ ]:
!pip uninstall -y torchao==0.10.0

## 0. Cấu hình & guard môi trường

Mọi tham số gom một chỗ (`DESIGN.md` §7). Ngưỡng verdict in ra **trước** khi chạy (văn hóa pre-registered của dự án). Guard `transformers<5` giữ nguyên pattern comment như nb7 (đã chạy pass không cần downgrade ngày 24–25/09).

In [ ]:
import os
import json
import random
import datetime
import unicodedata
from pathlib import Path
import collections

# Cài đặt thư viện cần thiết nếu chạy trên Kaggle (giống nb5/nb7)
try:
    import peft
    import sentencepiece
except ImportError:
    print('Cài đặt peft và sentencepiece...')
    os.system('pip install -q peft sentencepiece')
    os.system('pip uninstall -y torchao')

# Guard version: transformers v5 bug convert tokenizer sentencepiece (ViT5/T5-style) → KeyError: 0
# (nb7 đã chạy thành công với guard comment-out ngày 24–25/09 — giữ nguyên pattern)
import transformers
# from packaging.version import Version
# if Version(transformers.__version__).major >= 5:
#     print(f'transformers {transformers.__version__} (v5) — downgrade về <5...')
#     os.system('pip install -q "transformers<5"')
#     raise SystemExit(
#         'Đã downgrade transformers về <5. BÂY GIỜ: Restart Kernel '
#         '(Run → Restart & Clear Outputs) rồi Run All lại từ đầu — guard sẽ pass và chạy bình thường.'
#     )

import torch
from transformers import AutoTokenizer, set_seed
from peft import AutoPeftModelForSeq2SeqLM

SEED = 42
set_seed(SEED)

# --- Helper consts cho cell word acc (default của bootstrap_ci — copy giá trị nb5 §0) ---
N_RESAMPLE = 1000           # bootstrap CI (percentile 2.5 / 97.5)
BOOTSTRAP_ALPHA = 0.05

# --- Cấu hình nb8 ---
MODEL_NAME = 'vinai/bartpho-syllable'
EVALREAL_DATASET_ID = 'nrl-ai/vn-spell-correction-eval-real'
EVAL_BEAM_SIZE = 1          # greedy frozen (khớp protocol nb3/nb5/nb7)
EVAL_BATCH_SIZE = 16
MAX_SOURCE_LEN = 256        # khớp protocol nb3/nb5/nb7 (dùng trong batch_generate)
MAX_TARGET_LEN = 256
MIN_MARKED_TOKENS = 3       # guard ngữ cảnh: câu phải có >= 3 token chứa ký tự không-ASCII

# --- Ngưỡng verdict PRE-REGISTERED (plan 25/09 — chốt TRƯỚC khi nhìn kết quả) ---
VERDICT_MIN_WA_TELEX_GAIN = 0.05    # (i)   ΔWA telex_real (norm+PP − raw+PP) >= +5pp
VERDICT_MAX_OC_DELTA_TEST = 0.001   # (ii)  Δover-correction test (norm+PP − raw+PP) <= +0,1pp
VERDICT_MIN_F1_DELTA_TEST = -0.001  # (iii) ΔF1 test (norm+PP − raw+PP) >= −0,1pp
E3_MAX_CLEAN_TOUCHED = 0            # câu sạch test bị normalizer đụng phải = 0
MIN_TELEX_SLICE_N = 100             # slice telex test < 100 câu → verdict đọc kèm eval-real + ghi hạn chế

RUN2_REF = {  # nền tham chiếu REPORT.md §13 (test Run 2, seed 42, greedy) — chỉ để in đối chiếu
    'raw_f1': 0.8045, 'raw_overcorr': 0.0188, 'raw_clean_retention': 0.8591,
    'pp_f1': 0.8138, 'pp_overcorr': 0.0146, 'pp_clean_retention': 0.9648,
}

REQUIRED_FILES = [
    'predictions_val_run2.jsonl',    # nb3
    'predictions_test_run2.jsonl',   # nb3
    'postprocess_config.json',       # nb6 — frozen G3, KHÔNG re-tune
    'syllable_table.json',           # nb2 — bảng âm tiết (nguồn công khai độc lập)
    'test_aligned.jsonl',            # nb1 — đối chiếu text test
]
ADAPTER_DIRS = ['lora_adapter_run2']                        # nb3
OPTIONAL_FILES = ['predictions_evalreal_run2.jsonl']        # nb5 — raw preds eval-real
OUT_FILES = []


def find_required_inputs():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += (sorted(kaggle.rglob('*.jsonl')) + sorted(kaggle.rglob('*.json'))
                       + sorted(kaggle.rglob('adapter_config.json')))
    local = Path('./out')
    if local.is_dir():
        candidates += (sorted(local.rglob('*.jsonl')) + sorted(local.rglob('*.json'))
                       + sorted(local.rglob('adapter_config.json')))
    found = {}
    for p in candidates:
        name = p.name
        if name in REQUIRED_FILES or name in OPTIONAL_FILES:
            found.setdefault(name, p)
        elif name == 'adapter_config.json' and p.parent.name in ADAPTER_DIRS:
            found.setdefault(p.parent.name, p.parent)
    missing = [req for req in REQUIRED_FILES + ADAPTER_DIRS if req not in found]
    if missing:
        raise FileNotFoundError(
            'Thiếu input bắt buộc: ' + ', '.join(missing) +
            ' | nb8 cần Add Input: output nb3 (predictions_{val,test}_run2.jsonl + lora_adapter_run2/), '
            'output nb6 (postprocess_config.json), output nb2 (syllable_table.json), '
            'output nb1 (test_aligned.jsonl). Optional: output nb5 (predictions_evalreal_run2.jsonl — '
            'không có sẽ load eval-real từ HF và chạy lại nhánh raw, ghi deviation).'
        )
    return found


INPUT_FILES = find_required_inputs()
OPTIONAL_PRESENT = {k: INPUT_FILES[k] for k in OPTIONAL_FILES if k in INPUT_FILES}

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

FP16 = torch.cuda.is_available()
DEVICE = 'cuda' if FP16 else 'cpu'

print('=== NGƯỠNG VERDICT PRE-REGISTERED (in trước khi chạy) ===')
print(f'  (i)   ΔWA telex_real (norm+PP − raw+PP) >= +{VERDICT_MIN_WA_TELEX_GAIN:.1%}')
print(f'  (ii)  Δover-correction test (norm+PP − raw+PP) <= +{VERDICT_MAX_OC_DELTA_TEST:.1%}')
print(f'  (iii) ΔF1 test (norm+PP − raw+PP) >= {VERDICT_MIN_F1_DELTA_TEST:.1%}')
print(f'  E3: câu sạch test bị đụng = {E3_MAX_CLEAN_TOUCHED} · slice telex tối thiểu = {MIN_TELEX_SLICE_N} câu')
print(f'  Nền tham chiếu test Run 2 (REPORT §13): raw F1={RUN2_REF["raw_f1"]:.2%} → PP F1={RUN2_REF["pp_f1"]:.2%}')
print()
print('=== TỰ DÒ INPUT HOÀN TẤT ===')
for k in REQUIRED_FILES:
    print(f'  {k:32s}: {INPUT_FILES[k]}')
for k in ADAPTER_DIRS:
    print(f'  {k:32s}: {INPUT_FILES[k]}')
print('Optional (nb5, raw eval-real):', OPTIONAL_PRESENT if OPTIONAL_PRESENT
      else 'KHÔNG có → load eval-real từ HF + chạy lại nhánh raw (deviation, ghi report)')
print(f'Device: {"CUDA " + torch.cuda.get_device_name(0) if FP16 else "CPU (nb8 cần GPU T4 cho inference)"} | FP16={FP16}')

## 1. Cell hàm dùng chung — copy NGUYÊN VẸN từ nb5 (`align-v1`) + `evaluate_predictions` + helper word acc

Như nb3/nb5/nb6/nb7: shared cell của nb5 là bản hợp nhất đầy đủ (có `build_pseudo_annotation` + `SUSPECT_EDIT_RATIO`); hàm đánh giá 3 chỉ số và helper word acc / sentence exact / bootstrap CI copy nguyên vẹn từ nb5, không sửa một ký tự.

In [ ]:
import re

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')
_DIGIT_RE = re.compile(r'\d')

def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()

def canon_tokenize(s):
    """NFC + tách token: run chữ/số liền kề (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""
    return _TOKEN_RE.findall(nfc_normalize(s))

def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)

def is_word_token(tok):
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)

def levenshtein_opcodes(src, tgt):
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)
            if prev[j] + 1 < best:
                best = prev[j] + 1
            if row[j - 1] + 1 < best:
                best = row[j - 1] + 1
            row[j] = best
    ops = []
    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops

def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }

def extract_edit_blocks(src, tgt, opcodes):
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks

def apply_edit_blocks(src, blocks):
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out

SUSPECT_EDIT_RATIO = 0.3  # giữ nguyên giá trị nb1/nb2 — cell hàm dùng chung copy nguyên vẹn cần nó

def build_pseudo_annotation(text, corrected, suspect_ratio=SUSPECT_EDIT_RATIO):
    """Align text ↔ corrected_text trong hệ token canonical → pseudo-annotation schema giống VSEC + edit_blocks.
    Quy ước QUAN TRỌNG (các notebook sau phải dùng cell này nguyên vẹn):
    - error_positions: index token nguồn nằm trong src_span của block CÓ token nguồn.
      Block insert (thiếu âm tiết ở nguồn) KHÔNG có vị trí nguồn → không nằm trong error_positions,
      chỉ nằm trong correction_pairs với error='' và position = vị trí chèn trước (có thể == len(src)).
    - correction_pairs: 1 entry/block; error/correction = các token nối bằng space; delete → correction=''.
    - syllable_annotations: 1 entry/token nguồn; is_correct=False khi token thuộc src_span của block
      non-insert; corrections = chuỗi đích (join space) của block đó.
    - suspect: edit_ratio = (tổng token cả 2 vế nằm trong edit block) / max(len(src), len(tgt)) vượt ngưỡng.
    - align_failed: một trong hai vế token hóa rỗng."""
    src = canon_tokenize(text)
    tgt = canon_tokenize(corrected)
    blocks = extract_edit_blocks(src, tgt, levenshtein_opcodes(src, tgt))
    corrections_by_pos = {}
    for b in blocks:
        if b['src_span'][0] < b['src_span'][1]:
            fix = ' '.join(b['tgt_tokens'])
            for i in range(b['src_span'][0], b['src_span'][1]):
                corrections_by_pos.setdefault(i, []).append(fix)
    error_positions = sorted(corrections_by_pos)
    syllable_annotations = [
        {
            'syllable': tok,
            'is_correct': i not in corrections_by_pos,
            'corrections': corrections_by_pos.get(i, []),
            'position': i,
        }
        for i, tok in enumerate(src)
    ]
    correction_pairs = [
        {'error': ' '.join(b['src_tokens']), 'correction': ' '.join(b['tgt_tokens']), 'position': b['position']}
        for b in blocks
    ]
    n_edit_tokens = sum(
        (b['src_span'][1] - b['src_span'][0]) + (b['tgt_span'][1] - b['tgt_span'][0]) for b in blocks
    )
    denom = max(len(src), len(tgt))
    edit_ratio = n_edit_tokens / denom if denom else 0.0
    return {
        'is_clean': not blocks,
        'align_failed': not src or not tgt,
        'suspect': edit_ratio > suspect_ratio,
        'edit_ratio': round(edit_ratio, 4),
        'error_count': len(blocks),
        'error_positions': error_positions,
        'correction_pairs': correction_pairs,
        'syllable_annotations': syllable_annotations,
        'edit_blocks': blocks,
        'src_tokens': src,
        'tgt_tokens': tgt,
    }

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)

### `evaluate_predictions` — copy nguyên vẹn từ nb5 (giống nb3 §1)

Sanity case nhân tạo giữ nguyên. Chỉ số tính trên **text gốc** (gold không đổi) cho cả 2 nhánh — dự đoán của nhánh norm được align lại với câu gốc qua `align-v1`.

In [ ]:
def evaluate_predictions(records, predictions, syll_set=None):
    assert len(records) == len(predictions), f'Độ dài không khớp: {len(records)} vs {len(predictions)}'
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    correct_at_tp = 0
    
    total_clean_tokens = 0
    clean_sents_total = 0
    clean_sents_preserved = 0
    
    # Stratified stats: non-word vs real-word
    stratified = {
        'nonword': {'gold': 0, 'detected': 0, 'corrected': 0},
        'realword': {'gold': 0, 'detected': 0, 'corrected': 0}
    }
    
    sample_overcorrections = []
    
    for rec, pred in zip(records, predictions):
        src_toks = canon_tokenize(rec['text'])
        gold_toks = canon_tokenize(rec['corrected_text'])
        pred_toks = canon_tokenize(pred)
        
        # Gold edit blocks
        gold_ops = levenshtein_opcodes(src_toks, gold_toks)
        gold_blocks = extract_edit_blocks(src_toks, gold_toks, gold_ops)
        
        # Pred edit blocks
        pred_ops = levenshtein_opcodes(src_toks, pred_toks)
        pred_blocks = extract_edit_blocks(src_toks, pred_toks, pred_ops)
        
        gold_pos_map = {}
        for b in gold_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                target_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    gold_pos_map[p] = (target_str, b['src_tokens'])
                    
        pred_pos_map = {}
        for b in pred_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                pred_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    pred_pos_map[p] = (pred_str, b['src_tokens'])
                    
        gold_positions = set(gold_pos_map.keys())
        pred_positions = set(pred_pos_map.keys())
        
        tp_pos = gold_positions & pred_positions
        fp_pos = pred_positions - gold_positions
        fn_pos = gold_positions - pred_positions
        
        total_tp += len(tp_pos)
        total_fp += len(fp_pos)
        total_fn += len(fn_pos)
        
        # Đếm số token đúng trong câu nguồn (loại bỏ token dấu câu)
        word_token_positions = {i for i, t in enumerate(src_toks) if is_word_token(t)}
        clean_word_positions = word_token_positions - gold_positions
        total_clean_tokens += len(clean_word_positions)
        
        # Check clean sentence
        if len(gold_positions) == 0:
            clean_sents_total += 1
            if len(pred_positions) == 0:
                clean_sents_preserved += 1
                
        # Correction accuracy
        for p in tp_pos:
            target_str, _ = gold_pos_map[p]
            pred_str, _ = pred_pos_map[p]
            if pred_str.lower() == target_str.lower():
                correct_at_tp += 1
                
            # Stratified
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['detected'] += 1
                if pred_str.lower() == target_str.lower():
                    stratified[cat]['corrected'] += 1
                    
        for p in fn_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
        for p in tp_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
                
        # Lưu mẫu over-correction tiêu biểu
        if fp_pos and len(sample_overcorrections) < 20:
            for p in sorted(fp_pos):
                if p < len(src_toks):
                    sample_overcorrections.append({
                        'orig_token': src_toks[p],
                        'pred_token': pred_pos_map[p][0],
                        'context': ' '.join(src_toks[max(0, p-3):min(len(src_toks), p+4)]),
                        'full_src': rec['text'],
                        'full_pred': pred
                    })
                    if len(sample_overcorrections) >= 20:
                        break

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    corr_acc = correct_at_tp / total_tp if total_tp > 0 else 0.0
    overcorr_rate = total_fp / total_clean_tokens if total_clean_tokens > 0 else 0.0
    clean_retention = clean_sents_preserved / clean_sents_total if clean_sents_total > 0 else 1.0

    return {
        'detection': {
            'tp': total_tp,
            'fp': total_fp,
            'fn': total_fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        },
        'correction': {
            'correct_at_tp': correct_at_tp,
            'accuracy': corr_acc,
        },
        'over_correction': {
            'fp_count': total_fp,
            'clean_tokens': total_clean_tokens,
            'rate': overcorr_rate,
            'clean_sents_total': clean_sents_total,
            'clean_sents_preserved': clean_sents_preserved,
            'clean_retention_rate': clean_retention,
        },
        'stratified': stratified,
        'sample_overcorrections': sample_overcorrections,
    }

# Sanity Test hàm đánh giá trên ví dụ giả định (copy nguyên vẹn từ nb3)
_test_recs = [{'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}]
_test_preds = ['học sinh đi hoc'] # Model sửa 'sanh' -> 'sinh', nhưng bỏ sót 'hoc'
_m_test = evaluate_predictions(_test_recs, _test_preds)
assert _m_test['detection']['tp'] == 1 and _m_test['detection']['fn'] == 1 and _m_test['detection']['fp'] == 0
assert _m_test['correction']['accuracy'] == 1.0
print('Sanity check hàm evaluate_predictions PASS 100%!')

### Helper word accuracy / sentence exact / bootstrap CI — copy nguyên vẹn từ nb5

Word acc là **định nghĩa tự có của nb5** — chỉ so hướng (ordinal), kèm CI 95% (n=25/register thì CI rất rộng — đọc kèm).

In [ ]:
def word_acc(clean_text, pred_text):
    clean_toks = canon_tokenize(clean_text)
    pred_toks = canon_tokenize(pred_text)
    if not clean_toks:
        return 0.0
    ops = levenshtein_opcodes(clean_toks, pred_toks)
    equal_tokens = sum((i2 - i1) for tag, i1, i2, j1, j2 in ops if tag == 'equal')
    return equal_tokens / len(clean_toks)


def sentence_exact(clean_text, pred_text):
    return canon_tokenize(pred_text) == canon_tokenize(clean_text)


def bootstrap_ci(values, n_resample=N_RESAMPLE, alpha=BOOTSTRAP_ALPHA, seed=SEED):
    """CI percentile cho mean của list giá trị per-sentence (resample theo câu, seeded)."""
    if not values:
        return None, None
    rng = random.Random(seed)
    n = len(values)
    means = []
    for _ in range(n_resample):
        total = 0
        for _ in range(n):
            total += values[rng.randrange(n)]
        means.append(total / n)
    means.sort()
    m = len(means)                                    # số lần resample — không phải len(values)
    lo = means[max(0, int((alpha / 2) * m))]
    hi = means[min(m - 1, int((1 - alpha / 2) * m))]
    return lo, hi


def word_acc_block(pairs):
    """pairs: list (clean_text, pred_text) → stats word acc + sentence exact + CI."""
    accs = [word_acc(c, p) for c, p in pairs]
    exacts = [1.0 if sentence_exact(c, p) else 0.0 for c, p in pairs]
    lo, hi = bootstrap_ci(accs)
    lo_e, hi_e = bootstrap_ci(exacts)
    return {
        'n': len(pairs),
        'word_acc_mean': sum(accs) / len(accs) if accs else 0.0,
        'word_acc_ci95': [lo, hi],
        'sentence_exact_mean': sum(exacts) / len(exacts) if exacts else 0.0,
        'sentence_exact_ci95': [lo_e, hi_e],
    }


# Sanity checks
assert word_acc('học sinh đi học', 'học sinh đi hoc') == 3 / 4
assert word_acc('học sinh đi học', 'học sinh đi học') == 1.0
assert sentence_exact('a b', 'a b') and not sentence_exact('a b', 'a, b')
_lo, _hi = bootstrap_ci([1.0] * 50 + [0.0] * 50, n_resample=200)
assert _lo < 0.5 < _hi, (_lo, _hi)
print('Sanity helpers word_acc / sentence_exact / bootstrap_ci PASS')

## 2. Bảng đảo chuyển tự + normalizer + sanity nhân tạo

Hàm chuyển tự copy NGUYÊN VẸN từ nb7 §4 (`transliterate_style` + `telex_candidates`, kèm sanity `được → dduwowjc / d9u7o75c`). Bảng đảo: mỗi âm tiết ∈ bảng (nguồn công khai độc lập) → các dạng gõ telex/VNI của nó; token không hợp lệ có **đúng 1** expansion mới được áp (2 style cho cùng kết quả đếm là 1 — đã dedupe trong `telex_candidates`). Normalizer là hàm thuần deterministic: identity cho câu không đủ token có dấu / không có token đổi (nguyên vẹn từng bit); mọi token sau normalize giữ nguyên hoặc ∈ SYLL_SET; NFC assert.

Artifact cosmetic đã biết (như nb6): câu bị thay đổi được rebuild bằng `' '.join` — có thể mất multiple-space gốc; câu KHÔNG thay đổi trả về chuỗi gốc nguyên vẹn.

In [ ]:
SYLL_TABLE = json.loads(Path(INPUT_FILES['syllable_table.json']).read_text(encoding='utf-8'))
SYLL_SET = set(SYLL_TABLE['entries'])
print(f'Bảng âm tiết: {len(SYLL_SET)} entries')

_TONE_KEY_TELEX = {0x0301: 's', 0x0300: 'f', 0x0309: 'r', 0x0303: 'x', 0x0323: 'j'}
_TONE_KEY_VNI = {0x0301: '1', 0x0300: '2', 0x0309: '3', 0x0303: '4', 0x0323: '5'}
_CIRC, _BREVE, _HORN = 0x0302, 0x0306, 0x031B
_CIRC_KEY_TELEX = {'a': 'a', 'e': 'e', 'o': 'o'}
_CIRC_KEY_VNI = {'a': '6', 'e': '6', 'o': '6'}
_BREVE_KEY = {'telex': 'w', 'vni': '8'}
_HORN_KEY = {'telex': 'w', 'vni': '7'}


def transliterate_style(syl, style):
    """Chuyển tự tất định 1 âm tiết NFC sang chuỗi telex/VNI (đầu ra ASCII → NFC-safe)."""
    tone_key = _TONE_KEY_TELEX if style == 'telex' else _TONE_KEY_VNI
    circ_key = _CIRC_KEY_TELEX if style == 'telex' else _CIRC_KEY_VNI
    out = []
    for ch in syl:
        if ch == 'đ':
            out.append('dd' if style == 'telex' else 'd9')
            continue
        d = unicodedata.normalize('NFD', ch)
        base, suffix = d[0], ''
        for mk in d[1:]:
            o = ord(mk)
            if o == _CIRC:
                suffix += circ_key.get(base, '')
            elif o == _BREVE:
                suffix += _BREVE_KEY[style]
            elif o == _HORN:
                suffix += _HORN_KEY[style]
            else:
                suffix += tone_key.get(o, '')
        out.append(base + suffix)
    return ''.join(out)


def telex_candidates(low_syl):
    cands = []
    for style in ('telex', 'vni'):
        t = transliterate_style(low_syl, style)
        if t != low_syl and t not in cands:
            cands.append(t)
    return cands


# Sanity chuyển tự
assert transliterate_style('đã', 'telex') == 'ddax', transliterate_style('đã', 'telex')
assert transliterate_style('đã', 'vni') == 'd9a4', transliterate_style('đã', 'vni')
assert transliterate_style('trường', 'telex') == 'truwowfng', transliterate_style('trường', 'telex')  # ư→uw, ơ→ow
assert telex_candidates('ơ')
assert all(unicodedata.is_normalized('NFC', c) for c in telex_candidates('được'))
print('Sanity chuyển tự telex/VNI PASS:',
      transliterate_style('được', 'telex'), '/', transliterate_style('được', 'vni'))



def build_telex_reverse_table(syll_set):
    """Bảng đảo tất định: dạng gõ telex/VNI (thuần ASCII) → tập âm tiết gốc.
    Nguồn 100% công khai độc lập (bảng âm tiết nb2 + chuyển tự nb7) — không nhìn nhãn train/test (DESIGN.md §9)."""
    table = {}
    for s in syll_set:
        low = nfc_normalize(s).lower()
        for form in telex_candidates(low):
            table.setdefault(form, set()).add(low)
    for form, targets in table.items():
        assert all(ord(ch) < 128 for ch in form), form      # key luôn thuần ASCII (điều kiện (b) enforced cấu trúc)
        assert targets <= syll_set, form                    # mọi expansion ∈ bảng âm tiết (điều kiện validation)
    return table


TELEX_REVERSE = build_telex_reverse_table(SYLL_SET)
_n_multi = sum(1 for v in TELEX_REVERSE.values() if len(v) > 1)
_overlap = SYLL_SET & set(TELEX_REVERSE.keys())
print(f'Bảng đảo chuyển tự: {len(TELEX_REVERSE)} dạng gõ · {sum(len(v) for v in TELEX_REVERSE.values())} cặp form→âm tiết')
print(f'  ambiguous (>=2 âm tiết — sẽ bị bỏ qua): {_n_multi} · trùng âm tiết hợp lệ (điều kiện (a) chặn hết): {len(_overlap)}')


def telex_expand_token(low, syll_set, reverse):
    """Expansion duy nhất nếu low là dạng gõ telex/VNI không ambiguous, ngược lại None.
    Điều kiện pre-registered (a): low ∉ syll_set — từ đã hợp lệ không đụng, kể cả khi có expansion."""
    if low in syll_set:
        return None
    targets = reverse.get(low)
    if not targets or len(targets) != 1:
        return None                     # 0 hoặc >=2 candidate khác nhau → bỏ qua (bảo thủ, không sửa oan)
    return next(iter(targets))


def count_marked_tokens(toks):
    return sum(1 for t in toks if any(ord(ch) > 127 for ch in t))


def telex_normalize_text(text, syll_set, reverse, min_marked=MIN_MARKED_TOKENS):
    """Chuẩn hóa tất định 1 câu — hàm thuần (không random, không model):
    - Guard ngữ cảnh: câu phải có >= min_marked token chứa ký tự không-ASCII → nếu không: nguyên vẹn từng bit.
    - Token word: (a) lower ∉ syll_set, (b) đúng 1 expansion từ bảng đảo → áp (giữ hoa ký tự đầu);
      0 hoặc >=2 expansion → bỏ vị trí.
    - Invariant: token sau normalize hoặc giữ nguyên hoặc ∈ syll_set; NFC assert; không đổi → trả chuỗi gốc."""
    toks = canon_tokenize(text)
    if count_marked_tokens(toks) < min_marked:
        return text, []
    out = list(toks)
    changed = []
    for i, tok in enumerate(toks):
        if not is_word_token(tok):
            continue
        exp = telex_expand_token(nfc_normalize(tok).lower(), syll_set, reverse)
        if exp is None:
            continue
        out[i] = exp[:1].upper() + exp[1:] if tok[:1].isupper() else exp
        changed.append({'position': i, 'before': tok, 'after': out[i]})
    if not changed:
        return text, []                 # identity — nguyên vẹn từng bit
    norm_text = ' '.join(out)
    assert unicodedata.is_normalized('NFC', norm_text), text
    for a, b in zip(toks, out):
        assert b == a or b.lower() in syll_set, (a, b)
    return norm_text, changed


# --- Sanity nhân tạo (không đụng dữ liệu thật) ---
_expand = lambda low: telex_expand_token(low, SYLL_SET, TELEX_REVERSE)

# (1) Expansion telex cơ bản — sanity nb7 đã chứng minh chuyển tự đi ('được' → 'dduwowjc')
assert _expand('dduwowjc') == 'được', _expand('dduwowjc')
# (2) 'vowj'→'vợ': kiểm thực tế — nếu bảng không chứa 'vợ' thì GHI NHẬN, không ép (plan §nb8.2)
if 'vợ' in SYLL_SET:
    assert _expand('vowj') == 'vợ', _expand('vowj')
else:
    print('GHI NHẬN (plan §nb8.2): "vợ" không có trong bảng âm tiết — vowj không ra candidate hợp lệ, không ép.')
# (3) Token ngoài khả năng chuyển tự → None (URL/ten riêng/từ viết tắt)
for tok in ('www', 'bhxh', 'honda', 'online'):
    assert _expand(tok) is None, tok
assert _expand('hoa') is None
# (4) Toàn bảng: mọi âm tiết hợp lệ đều trả None (điều kiện (a) enforced cấu trúc trên cả bảng)
assert all(telex_expand_token(s, SYLL_SET, TELEX_REVERSE) is None for s in SYLL_SET)

# (5) Mức câu: guard ngữ cảnh + identity từng bit
_t = 'xem tai www.x.com ngay hom nay luon'
_t2, _c = telex_normalize_text(_t, SYLL_SET, TELEX_REVERSE)
assert _t2 == _t and _c == []                       # <3 token có dấu → không đụng
_t = 'Honda BHXH online hôm qua rồi đó'
_t2, _c = telex_normalize_text(_t, SYLL_SET, TELEX_REVERSE)
assert _t2 == _t and _c == [], (_t2, _c)            # token lạ không đụng dù câu đủ token có dấu
# (6) Mức câu: expansion đúng, giữ hoa ký tự đầu
if 'vợ' in SYLL_SET:
    _t = 'Nhà vowj cửa học bật sổ'
    _t2, _c = telex_normalize_text(_t, SYLL_SET, TELEX_REVERSE)
    assert _t2 == 'Nhà vợ cửa học bật sổ' and len(_c) == 1, (_t2, _c)
_t = 'Tôi ddax học bài rồi'
_t2, _c = telex_normalize_text(_t, SYLL_SET, TELEX_REVERSE)
assert _t2 == 'Tôi đã học bài rồi' and len(_c) == 1, (_t2, _c)
# (7) Deterministic: gọi 2 lần ra y hệt
assert telex_normalize_text('Tôi ddax học bài rồi', SYLL_SET, TELEX_REVERSE) == \
       telex_normalize_text('Tôi ddax học bài rồi', SYLL_SET, TELEX_REVERSE)
print('Sanity normalizer PASS: dduwowjc→được · hoa/www/BHXH/Honda không đụng · identity từng bit · '
      'guard ngữ cảnh · deterministic')


## 3. Dữ liệu + thống kê tác động + E3 guard ablation

Nhánh raw dùng lại file nb3 (val/test) + nb5 (eval-real) — không chạy lại. Thống kê trước khi tốn GPU: % câu bị normalizer đổi trên val/test/eval-real, số token đổi, phân bố theo style (telex / VNI / cả hai cho cùng kết quả); **E3**: câu sạch (0 gold edit qua pseudo-annotation) bị đụng — mục tiêu **0**; FP phía token hợp lệ = 0 theo cấu trúc (assert). Slice telex = câu bị đổi ≥1 token (gold-agnostic — dùng cho F1 slice ở §5).

In [ ]:
def load_prediction_set(path):
    recs = load_jsonl(path)
    preds = [r['prediction'] for r in recs]
    assert len(recs) == len(preds)
    for r in recs:
        assert 'text' in r and 'corrected_text' in r and 'prediction' in r, r.get('row_id')
    return recs, preds

val_recs, val_preds_raw = load_prediction_set(INPUT_FILES['predictions_val_run2.jsonl'])
test_recs, test_preds_raw = load_prediction_set(INPUT_FILES['predictions_test_run2.jsonl'])

# Đối chiếu test_run2 ↔ test_aligned.jsonl (như nb6 — bảo đảm cùng nguồn, cùng thứ tự)
_test_aligned = load_jsonl(INPUT_FILES['test_aligned.jsonl'])
assert len(test_recs) == len(_test_aligned), (len(test_recs), len(_test_aligned))
_mismatch = sum(1 for r, t in zip(test_recs, _test_aligned)
                if r['text'] != t['text'] or r['corrected_text'] != t['corrected_text'])
assert _mismatch == 0, f'test_run2: {_mismatch} dòng lệch text/corrected_text vs test_aligned.jsonl'
print('Đối chiếu test_run2 ↔ test_aligned.jsonl: KHỚP 100% (text + corrected_text, cùng thứ tự)')

# eval-real: ưu tiên file nb5 (raw preds frozen — REPORT §12); không có → load HF (cần Internet ON)
RAW_EVALREAL_SOURCE = None
bench_meta = None
evalreal_recs, evalreal_preds_raw = None, None
if 'predictions_evalreal_run2.jsonl' in OPTIONAL_PRESENT:
    evalreal_recs, evalreal_preds_raw = load_prediction_set(OPTIONAL_PRESENT['predictions_evalreal_run2.jsonl'])
    for r in evalreal_recs:
        assert 'register' in r, 'File nb5 predictions_evalreal_run2.jsonl thiếu register — kiểm tra lại output nb5'
    RAW_EVALREAL_SOURCE = 'file nb5 (predictions_evalreal_run2.jsonl) — raw preds frozen, khớp REPORT §12'
    print(f'eval-real: {len(evalreal_recs)} cặp từ file nb5 · raw preds dùng lại (không chạy lại)')
else:
    print('== Load eval-real từ HF (cần Internet ON) — khối dưới copy từ nb5 §2 (thụt lề để nằm trong nhánh guard) ==')
    RAW_EVALREAL_SOURCE = ('RERUN raw trong phiên này (deviation ghi report — thiếu file nb5; '
                           '2 nhánh cùng GPU session nên so sánh vẫn fair)')

if evalreal_recs is None:

    from datasets import load_dataset, get_dataset_config_names
    import datasets

    EVALREAL_FALLBACK_HINT = (
        'Không tải được ' + EVALREAL_DATASET_ID + '. '
        'Trên Kaggle: bật Internet (Settings → Internet ON) rồi chạy lại. '
        'Hoặc fallback thủ công: download dataset từ Hugging Face, nén thành Kaggle Dataset '
        'giữ cấu trúc <config>/ chứa file data (parquet/jsonl/csv), Add Input vào notebook này.'
    )


    def extract_pair(ex):
        key_pairs = [('noisy', 'clean'), ('noisy_text', 'clean_text'), ('text', 'clean'),
                     ('text', 'label'), ('input', 'target'), ('sentence', 'correction')]
        for nk, ck in key_pairs:
            if nk in ex and ck in ex:
                return str(ex[nk]), str(ex[ck])
        raise KeyError(f'Không nhận diện được cặp trường noisy/clean trong example: keys={list(ex.keys())}')


    evalreal_records = []
    bench_meta = {'dataset_id': EVALREAL_DATASET_ID, 'datasets_version': datasets.__version__, 'configs': {}}
    try:
        config_names = sorted(get_dataset_config_names(EVALREAL_DATASET_ID))
        print('Configs phát hiện:', config_names)
        for cfg in config_names:
            ds = load_dataset(EVALREAL_DATASET_ID, cfg, split='train')
            first_keys = None
            for ex in ds:
                noisy, clean = extract_pair(ex)
                if first_keys is None:
                    first_keys = list(ex.keys())
                evalreal_records.append({
                    'row_id': f'{cfg}#{len(evalreal_records):04d}',
                    'register': cfg,
                    'text': nfc_normalize(str(noisy)),
                    'corrected_text': nfc_normalize(str(clean)),
                })
            bench_meta['configs'][cfg] = {'n': len(ds), 'example_keys': first_keys}
            print(f'  {cfg:16s}: {len(ds)} cặp — keys={first_keys}')
    except Exception as exc:
        raise RuntimeError(EVALREAL_FALLBACK_HINT + f' | Chi tiết lỗi: {exc}') from exc

    print(f'Tổng: {len(evalreal_records)} cặp (kỳ vọng ~150, 6 register)')
    assert len(evalreal_records) > 0, 'eval-real rỗng — dừng thay vì đo trên bench rỗng'

    evalreal_recs = evalreal_records

print()


def normalize_dataset(records):
    texts_norm, changed_stats = [], []
    for r in records:
        t, ch = telex_normalize_text(r['text'], SYLL_SET, TELEX_REVERSE)
        texts_norm.append(t)
        changed_stats.append(ch)
    return texts_norm, changed_stats


def style_of_change(before_tok, after_tok):
    """Style sinh ra dạng gõ của token bị đổi: telex / vni / telex+vni (cả 2 cùng kết quả — đếm một lần khi dedupe)."""
    low_exp = nfc_normalize(after_tok).lower()
    styles = []
    if transliterate_style(low_exp, 'telex') == before_tok.lower():
        styles.append('telex')
    if transliterate_style(low_exp, 'vni') == before_tok.lower():
        styles.append('vni')
    return '+'.join(styles) if styles else 'không-khớp'


stats_impact = {}
for name, recs in [('val', val_recs), ('test', test_recs), ('evalreal', evalreal_recs)]:
    texts_norm, changed = normalize_dataset(recs)
    n_changed_sents = sum(1 for c in changed if c)
    n_tokens = sum(len(c) for c in changed)
    styles = collections.Counter(style_of_change(x['before'], x['after']) for c in changed for x in c)
    clean_total = clean_touched = touched_valid_token = 0
    for r, c in zip(recs, changed):
        is_clean = not build_pseudo_annotation(r['text'], r['corrected_text'])['error_positions']
        clean_total += 1 if is_clean else 0
        if c and is_clean:
            clean_touched += 1
        for x in c:   # không thể xảy ra theo cấu trúc — assert cứng (FP phía token hợp lệ = 0)
            if nfc_normalize(x['before']).lower() in SYLL_SET:
                touched_valid_token += 1
    assert touched_valid_token == 0, 'Invariant (a) vi phạm: token hợp lệ bị đụng!'
    stats_impact[name] = {
        'n_sentences': len(recs), 'n_sentences_changed': n_changed_sents,
        'pct_sentences_changed': n_changed_sents / len(recs),
        'n_tokens_changed': n_tokens, 'style_breakdown': dict(styles),
        'e3_clean_total': clean_total, 'e3_clean_touched': clean_touched,
        'texts_norm': texts_norm, 'changed': changed,
    }
    print(f'{name:9s}: {n_changed_sents}/{len(recs)} câu đổi ({n_changed_sents / len(recs):.2%}) · '
          f'{n_tokens} token · style {dict(styles)} · E3 sạch {clean_touched}/{clean_total} (mục tiêu 0)')

E3_CLEAN_TOUCHED_TEST = stats_impact['test']['e3_clean_touched']
E3_OK = E3_CLEAN_TOUCHED_TEST <= E3_MAX_CLEAN_TOUCHED
TELEX_SLICE_TEST = [i for i, c in enumerate(stats_impact['test']['changed']) if c]
TELEX_SLICE_VAL = [i for i, c in enumerate(stats_impact['val']['changed']) if c]
print(f'E3 test: {E3_CLEAN_TOUCHED_TEST}/{stats_impact["test"]["e3_clean_total"]} câu sạch bị đụng '
      f'(ngưỡng pre-registered <= {E3_MAX_CLEAN_TOUCHED}) → {"PASS" if E3_OK else "ĐIỀU TRA — không âm thầm nới ngưỡng"}')
print(f'Slice telex: test {len(TELEX_SLICE_TEST)} câu · val {len(TELEX_SLICE_VAL)} câu '
      f'(ngưỡng đọc độc: >= {MIN_TELEX_SLICE_N})')
print('Mẫu test bị đổi (5 đầu):')
for _i in TELEX_SLICE_TEST[:5]:
    print('  -', test_recs[_i]['text'][:100])
    print('   →', stats_impact['test']['texts_norm'][_i][:100])

# --- Ghi sớm telex_normalizer_rules.json (detector params + thống kê — rẻ, ghi trước khi tốn GPU) ---
normalizer_rules = {
    'created': RUN_STAMP,
    'notebook': 'nb8_telex_normalization',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'seed': SEED,
    'detector': {
        'min_marked_tokens': MIN_MARKED_TOKENS,
        'eligibility': '(a) lower ∉ SYLL_SET · (b) thuần ASCII (bảng đảo key ASCII) · '
                       '(c) đúng 1 expansion từ bảng đảo (dedupe telex+VNI)',
    },
    'resources': {
        'source': 'syllable_table.json (nb2, nguồn công khai độc lập) × transliterate_style/telex_candidates (nb7 §4, verbatim)',
        'syllable_table_n': len(SYLL_SET),
        'reverse_table_forms': len(TELEX_REVERSE),
        'reverse_table_ambiguous_forms': _n_multi,
        'overlap_valid_syllable_in_keys': len(_overlap),
    },
    'impact': {k: {kk: vv for kk, vv in stats_impact[k].items() if kk not in ('texts_norm', 'changed')}
               for k in stats_impact},
    'e3': {'clean_touched_test': E3_CLEAN_TOUCHED_TEST, 'target': E3_MAX_CLEAN_TOUCHED, 'pass': E3_OK},
    'telex_slice': {'test_n': len(TELEX_SLICE_TEST), 'val_n': len(TELEX_SLICE_VAL),
                    'min_for_standalone_verdict': MIN_TELEX_SLICE_N},
}
with open(OUTPUT_DIR / 'telex_normalizer_rules.json', 'w', encoding='utf-8') as f:
    json.dump(normalizer_rules, f, ensure_ascii=False, indent=2)
OUT_FILES.append('telex_normalizer_rules.json')
print('Đã ghi telex_normalizer_rules.json')

## 4. Inference nhánh chuẩn hóa (T4 · greedy — protocol nb3/nb5/nb7)

`batch_generate` copy nguyên vẹn nb7 §6, `load_adapter` copy nguyên vẹn nb5 §3. Chạy đầu vào đã chuẩn hóa qua adapter Run 2 frozen. Nhánh raw eval-real: dùng file nb5 nếu có; không có → chạy lại raw trong phiên này (deviation — 2 nhánh cùng session nên vẫn fair).

In [ ]:
def batch_generate(model, tokenizer, texts, batch_size=16, beam_size=EVAL_BEAM_SIZE):  # copy nb3 §7
    model.eval()
    device = next(model.parameters()).device
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True,
                        max_length=MAX_SOURCE_LEN, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                max_length=MAX_TARGET_LEN,
                num_beams=beam_size,
                early_stopping=True if beam_size > 1 else False
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        preds.extend([nfc_normalize(d) for d in decoded])
        if (i // batch_size) % 50 == 0:
            print(f'  Generated {min(i + batch_size, len(texts))}/{len(texts)} sentences...')
    return preds

def load_adapter(adapter_dir):
    print(f'Nạp adapter: {adapter_dir}')
    model = AutoPeftModelForSeq2SeqLM.from_pretrained(
        str(adapter_dir), torch_dtype=torch.float16 if FP16 else torch.float32
    ).to(DEVICE)
    model.eval()
    return model



def save_norm_jsonl(recs, texts_norm, changed, preds, out_path):
    """Schema nb3 + input_normalized + n_input_tokens_changed."""
    with open(out_path, 'w', encoding='utf-8') as f:
        for r, t, c, p in zip(recs, texts_norm, changed, preds):
            f.write(json.dumps({'row_id': r.get('row_id'), 'text': r['text'],
                                'corrected_text': r['corrected_text'], 'prediction': p,
                                'input_normalized': t, 'n_input_tokens_changed': len(c)},
                               ensure_ascii=False) + '\n')


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer BARTpho sẵn sàng')

model = load_adapter(INPUT_FILES['lora_adapter_run2'])

print('--- INFERENCE NORM · val ---')
preds_val_norm = batch_generate(model, tokenizer, stats_impact['val']['texts_norm'])
print('--- INFERENCE NORM · test 6k ---')
preds_test_norm = batch_generate(model, tokenizer, stats_impact['test']['texts_norm'])
print('--- INFERENCE NORM · eval-real ---')
preds_evalreal_norm = batch_generate(model, tokenizer, stats_impact['evalreal']['texts_norm'])

if evalreal_preds_raw is None:
    print('--- INFERENCE RAW · eval-real (RERUN — deviation do thiếu file nb5) ---')
    evalreal_preds_raw = batch_generate(model, tokenizer, [r['text'] for r in evalreal_recs])

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_n_empty = sum(1 for p in preds_val_norm + preds_test_norm + preds_evalreal_norm if not p.strip())
print(f'Inference hoàn tất · empty preds (nhánh norm) = {_n_empty}')

save_norm_jsonl(val_recs, stats_impact['val']['texts_norm'], stats_impact['val']['changed'],
                preds_val_norm, OUTPUT_DIR / 'predictions_val_run2_norm.jsonl')
save_norm_jsonl(test_recs, stats_impact['test']['texts_norm'], stats_impact['test']['changed'],
                preds_test_norm, OUTPUT_DIR / 'predictions_test_run2_norm.jsonl')
save_norm_jsonl(evalreal_recs, stats_impact['evalreal']['texts_norm'], stats_impact['evalreal']['changed'],
                preds_evalreal_norm, OUTPUT_DIR / 'predictions_evalreal_run2_norm.jsonl')
OUT_FILES += ['predictions_val_run2_norm.jsonl', 'predictions_test_run2_norm.jsonl',
              'predictions_evalreal_run2_norm.jsonl']
print('Đã ghi 3 file predictions_*_norm.jsonl')

## 5. Post-processing frozen G3 (nb6) + bảng A/B

Nạp `postprocess_config.json` của nb6 — **frozen, KHÔNG re-tune** (assert tên config `G3-r1r2-r3k1`; `resources.r3_whitelist` nạp về set). Rule engine R1–R4 copy nguyên vẹn nb6 §2 (`veto_reason` / `postprocess_one` / `postprocess_preds`). Áp cả 2 nhánh → bảng A/B: F1 / CorrAcc / over-corr / CleanRet trên val / test / eval-real + WA per-register + F1 slice telex test. Nhánh norm: PP align pred_norm với **input đã chuẩn hóa** (text model nhìn thấy); chỉ số vẫn tính trên text gốc.

In [ ]:
def veto_reason(block, cfg, res, syll_set):
    """Trả về tên rule nếu block bị phủ quyết (gold-agnostic khi áp: chỉ nhìn block shape + resource từ val)."""
    if block['src_span'][0] >= block['src_span'][1]:
        return None  # insert block: không hoàn nguyên được
    if cfg.get('r1_punct_guard') and block['punct_only']:
        return 'R1'
    is_pure_delete = block['tgt_span'][0] == block['tgt_span'][1]
    if is_pure_delete:
        low_toks = [nfc_normalize(t).lower() for t in block['src_tokens']]
        if cfg.get('r2_veto_valid_syllable') and low_toks and all(t in syll_set for t in low_toks):
            return 'R2'
        if cfg.get('r3_whitelist_min_count', 0) > 0 and low_toks \
                and all(t in res['r3_whitelist'] for t in low_toks):
            return 'R3'
    if cfg.get('r4_phrase_blacklist') and len(block['src_tokens']) >= 2:
        phrase = ' '.join(nfc_normalize(t).lower() for t in block['src_tokens'])
        if phrase in res['r4_phrases']:
            return 'R4'
    return None


def postprocess_one(text, pred, cfg, res, syll_set):
    """Áp rule lên 1 cặp (text, pred): veto block → hoàn nguyên tgt = src, rebuild bằng apply_edit_blocks.
    Không block nào bị veto → trả về prediction gốc NGUYÊN VẸN (identity guarantee)."""
    src = canon_tokenize(text)
    ptoks = canon_tokenize(pred)
    blocks = extract_edit_blocks(src, ptoks, levenshtein_opcodes(src, ptoks))
    hits = collections.Counter()
    changed = False
    for b in blocks:
        reason = veto_reason(b, cfg, res, syll_set)
        if reason:
            b['tgt_tokens'] = list(b['src_tokens'])
            b['tgt_span'] = list(b['src_span'])
            hits[reason] += 1
            changed = True
    if not changed:
        return pred, hits
    out_toks = apply_edit_blocks(src, blocks)
    out = ' '.join(out_toks)
    assert canon_tokenize(out) == out_toks, pred          # roundtrip token hóa
    assert unicodedata.is_normalized('NFC', out), pred    # NFC invariant
    return out, hits


def postprocess_preds(records, preds, cfg, res, syll_set):
    outs, hit_counts = [], collections.Counter()
    n_changed = 0
    for rec, pred in zip(records, preds):
        out, hits = postprocess_one(rec['text'], pred, cfg, res, syll_set)
        outs.append(out)
        if out != pred:
            n_changed += 1
        hit_counts.update(hits)
    return outs, {'n_changed_sentences': n_changed, 'veto_by_rule': dict(hit_counts)}


# --- Sanity case nhân tạo (không dùng dữ liệu thật) ---
_syll = {'hòa', 'học', 'sinh', 'an', 'tòa'}
_res1 = {'r3_whitelist': {'xyzq'}, 'r4_phrases': {'biểm họa'}}
_cfg_r12 = {'r1_punct_guard': True, 'r2_veto_valid_syllable': True,
            'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False}

# identity prediction → nguyên vẹn
_t, _h = postprocess_one('học sinh hòa an', 'học sinh hòa an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh hòa an' and not _h
# R2: xóa 'hòa' (âm tiết hợp lệ) → hoàn tác
_t, _h = postprocess_one('học sinh hòa an', 'học sinh an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh hòa an' and _h['R2'] == 1, (_t, _h)
# R2 off → giữ nguyên xóa
_cfg_no = dict(_cfg_r12, r2_veto_valid_syllable=False)
_t, _h = postprocess_one('học sinh hòa an', 'học sinh an', _cfg_no, _res1, _syll)
assert _t == 'học sinh an' and not _h
# xóa non-word ngoài whitelist → R2 KHÔNG bắn (không hoàn tác oan), giữ xóa
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r12, _res1, _syll)
assert _t == 'học an' and not _h
# R3: token trong whitelist (k=1) → hoàn tác
_cfg_r13 = dict(_cfg_r12, r3_whitelist_min_count=1)
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r13, _res1, _syll)
assert _t == 'học xyzq an' and _h['R3'] == 1, (_t, _h)
# grid k=0 (không whitelist) → giữ xóa
_cfg_r2only = {'r1_punct_guard': False, 'r2_veto_valid_syllable': False,
               'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False}
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r2only, _res1, _syll)
assert _t == 'học an' and not _h
# R1: xóa dấu câu → hoàn tác
_t, _h = postprocess_one('học sinh , an', 'học sinh an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh , an' and _h['R1'] == 1, (_t, _h)
# R4: phrase blacklist 2-token → hoàn tác cả block
_cfg_r4 = {'r1_punct_guard': False, 'r2_veto_valid_syllable': False,
           'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': True}
_res4 = {'r3_whitelist': set(), 'r4_phrases': {'biểm họa'}}
_t, _h = postprocess_one('biểm họa lan', 'lan', _cfg_r4, _res4, _syll)
assert _t == 'biểm họa lan' and _h['R4'] == 1, (_t, _h)
print('Sanity rule engine PASS (identity · R1 · R2 on/off · R2 miss non-word · R3 whitelist · R4 phrase)')


# --- Nạp frozen config nb6 (KHÔNG re-tune) ---
_pp_cfg = json.loads(Path(INPUT_FILES['postprocess_config.json']).read_text(encoding='utf-8'))
PP_NAME = _pp_cfg['chosen_config_name']
assert PP_NAME == 'G3-r1r2-r3k1', f'Config không phải G3-r1r2-r3k1 frozen của nb6: {PP_NAME} — dừng, không tự re-tune.'
PP_CFG = dict(_pp_cfg['chosen_config'])
PP_RES = {
    'r3_whitelist': {t for t, _n in _pp_cfg['resources']['r3_whitelist']},      # list cặp [token, count] → set token
    'r4_phrases': {p for p, _n in _pp_cfg['resources'].get('r4_phrase_blacklist', [])},
}
print(f'PP frozen: {PP_NAME} = {PP_CFG} · whitelist {len(PP_RES["r3_whitelist"])} token '
      f'(nguồn: {_pp_cfg["resources"]["source"]})')
if len(PP_RES['r3_whitelist']) == 0:
    print('CẢNH BÁO: whitelist rỗng — kiểm tra postprocess_config.json')
if _pp_cfg['resources'].get('syllable_table_n') not in (None, len(SYLL_SET)):
    print(f'CẢNH BÁO: syllable_table_n nb6={_pp_cfg["resources"]["syllable_table_n"]} != bảng hiện tại {len(SYLL_SET)}')


def run_branch(recs, texts_for_pp, preds):
    """PP với text nền tương ứng nhánh: raw → text gốc; norm → text đã chuẩn hóa (input model nhìn thấy)."""
    return postprocess_preds([{'text': t, 'corrected_text': r['corrected_text']}
                              for r, t in zip(recs, texts_for_pp)],
                             preds, PP_CFG, PP_RES, SYLL_SET)


AB = {}
for name, recs, raw_preds, norm_preds in [('val', val_recs, val_preds_raw, preds_val_norm),
                                          ('test', test_recs, test_preds_raw, preds_test_norm),
                                          ('evalreal', evalreal_recs, evalreal_preds_raw, preds_evalreal_norm)]:
    texts_norm = stats_impact[name]['texts_norm']
    raw_pp, raw_pp_stats = run_branch(recs, [r['text'] for r in recs], raw_preds)
    norm_pp, norm_pp_stats = run_branch(recs, texts_norm, norm_preds)
    AB[name] = {
        'raw': evaluate_predictions(recs, raw_preds, SYLL_SET),
        'raw_pp': evaluate_predictions(recs, raw_pp, SYLL_SET),
        'norm': evaluate_predictions(recs, norm_preds, SYLL_SET),
        'norm_pp': evaluate_predictions(recs, norm_pp, SYLL_SET),
        'raw_preds': raw_preds, 'raw_pp_preds': raw_pp,
        'norm_preds': norm_preds, 'norm_pp_preds': norm_pp,
        'raw_pp_stats': raw_pp_stats, 'norm_pp_stats': norm_pp_stats,
    }

_registers = sorted({r['register'] for r in evalreal_recs})
assert 'telex_real' in _registers, f'thiếu register telex_real trong eval-real: {_registers}'


def _row(label, m):
    return (f"{label:>10s} | {m['detection']['precision']:>7.2%} | {m['detection']['recall']:>7.2%} | "
            f"{m['detection']['f1']:>7.2%} | {m['correction']['accuracy']:>8.2%} | "
            f"{m['over_correction']['rate']:>9.2%} | {m['over_correction']['clean_retention_rate']:>8.2%}")


print('=' * 94)
print(f"  {'set':<9s} | {'branch':>10s} | {'Prec':>7s} | {'Rec':>7s} | {'F1':>7s} | {'CorrAcc':>8s} | {'Over-corr':>9s} | {'CleanRet':>8s}")
print('-' * 94)
for _name in ('val', 'test', 'evalreal'):
    for _br in ('raw', 'raw_pp', 'norm', 'norm_pp'):
        print(f'  {_name:<9s} | ' + _row(_br, AB[_name][_br]))
    _a, _b = AB[_name]['raw_pp'], AB[_name]['norm_pp']
    print(f'  {_name:<9s} |   Δ(norm+PP − raw+PP): '
          f'F1 {_b["detection"]["f1"] - _a["detection"]["f1"]:+.2%} · '
          f'CorrAcc {_b["correction"]["accuracy"] - _a["correction"]["accuracy"]:+.2%} · '
          f'OC {_b["over_correction"]["rate"] - _a["over_correction"]["rate"]:+.2%} · '
          f'CleanRet {_b["over_correction"]["clean_retention_rate"] - _a["over_correction"]["clean_retention_rate"]:+.2%}')
    print('-' * 94)
print('=' * 94)

# --- F1 slice telex (slice xác định bằng chính detector — gold-agnostic) ---
DELTA_F1_SLICE = None
if TELEX_SLICE_TEST:
    _sub = [test_recs[i] for i in TELEX_SLICE_TEST]
    _f1_raw = evaluate_predictions(_sub, [AB['test']['raw_pp_preds'][i] for i in TELEX_SLICE_TEST],
                                   SYLL_SET)['detection']['f1']
    _f1_norm = evaluate_predictions(_sub, [AB['test']['norm_pp_preds'][i] for i in TELEX_SLICE_TEST],
                                    SYLL_SET)['detection']['f1']
    DELTA_F1_SLICE = _f1_norm - _f1_raw
    print(f'Slice telex test (n={len(TELEX_SLICE_TEST)}): F1 raw+PP {_f1_raw:.2%} → norm+PP {_f1_norm:.2%} '
          f'(Δ {DELTA_F1_SLICE:+.2%})')
else:
    print('Slice telex test RỖNG — detector không đổi câu nào trên test.')

# --- Word acc eval-real: raw+PP vs norm+PP (copy semantics nb5 §6) ---
wa_evalreal = {}
for _br, _preds in [('raw', AB['evalreal']['raw_preds']), ('raw_pp', AB['evalreal']['raw_pp_preds']),
                    ('norm', AB['evalreal']['norm_preds']), ('norm_pp', AB['evalreal']['norm_pp_preds'])]:
    wa_evalreal[_br] = {
        'overall': word_acc_block([(r['corrected_text'], p) for r, p in zip(evalreal_recs, _preds)]),
        'per_register': {},
    }
    for _cfg in _registers:
        _idx = [i for i, r in enumerate(evalreal_recs) if r['register'] == _cfg]
        wa_evalreal[_br]['per_register'][_cfg] = word_acc_block(
            [(evalreal_recs[i]['corrected_text'], _preds[i]) for i in _idx])

_fmt = lambda blk: f"{blk['word_acc_mean']:.2%} [{blk['word_acc_ci95'][0]:.2%}–{blk['word_acc_ci95'][1]:.2%}]"
print()
print('=== WORD ACC · eval-real (mean [CI95]) — headline: norm+PP vs raw+PP ===')
print(f"  {'register':<14s} | {'raw+PP':>27s} | {'norm+PP':>27s} | {'Δ':>8s}")
for _cfg in _registers:
    _r = wa_evalreal['raw_pp']['per_register'][_cfg]['word_acc_mean']
    _n = wa_evalreal['norm_pp']['per_register'][_cfg]['word_acc_mean']
    print(f'  {_cfg:<14s} | {_fmt(wa_evalreal["raw_pp"]["per_register"][_cfg]):>27s} | '
          f'{_fmt(wa_evalreal["norm_pp"]["per_register"][_cfg]):>27s} | {_n - _r:>+8.2%}')
_r = wa_evalreal['raw_pp']['overall']['word_acc_mean']
_n = wa_evalreal['norm_pp']['overall']['word_acc_mean']
print(f'  {"OVERALL":<14s} | {_fmt(wa_evalreal["raw_pp"]["overall"]):>27s} | '
      f'{_fmt(wa_evalreal["norm_pp"]["overall"]):>27s} | {_n - _r:>+8.2%}')

DELTA_WA_TELEX = (wa_evalreal['norm_pp']['per_register']['telex_real']['word_acc_mean']
                  - wa_evalreal['raw_pp']['per_register']['telex_real']['word_acc_mean'])
print(f'>> ΔWA telex_real (norm+PP − raw+PP) = {DELTA_WA_TELEX:+.2%} — ngưỡng (i) >= +{VERDICT_MIN_WA_TELEX_GAIN:.1%} '
      '(n=25/register — CI rất rộng, chỉ đọc hướng kèm CI)')


## 6. Verdict pre-registered + QA + xuất file

Verdict chỉ theo ngưỡng đã in ở §0 (không nới ngưỡng). QA: 10 mẫu test đầu tiên bị normalizer đổi (SRC / SRC_NORM / GOLD / PRED raw+PP / PRED norm+PP). Lưu ý QA: nhánh PP rebuild bằng `' '.join` có thể hiện khoảng trắng trước dấu câu (`Sê San.` → `Sê San .`) — artifact đã biết của post-processing, không ảnh hưởng metric (canon_tokenize tách punct sẵn).

In [ ]:
_a, _b = AB['test']['raw_pp'], AB['test']['norm_pp']
DELTA_F1_TEST = _b['detection']['f1'] - _a['detection']['f1']
DELTA_OC_TEST = _b['over_correction']['rate'] - _a['over_correction']['rate']

checks = {
    'i_wa_telex_real_ge_5pp': DELTA_WA_TELEX >= VERDICT_MIN_WA_TELEX_GAIN,
    'ii_oc_test_le_plus01pp': DELTA_OC_TEST <= VERDICT_MAX_OC_DELTA_TEST,
    'iii_f1_test_ge_minus01pp': DELTA_F1_TEST >= VERDICT_MIN_F1_DELTA_TEST,
}
PASS_HUONG_A = all(checks.values()) and E3_OK
_slice_note = ('đủ lớn — verdict đọc độc được' if len(TELEX_SLICE_TEST) >= MIN_TELEX_SLICE_N else
               f'NHỎ HƠN {MIN_TELEX_SLICE_N} — verdict đọc kèm eval-real + ghi hạn chế (plan §rủi ro)')

print('=== VERDICT PRE-REGISTERED (ngưỡng đã chốt ở §0 — không nới) ===')
print(f'  (i)   ΔWA telex_real = {DELTA_WA_TELEX:+.2%} (>= +{VERDICT_MIN_WA_TELEX_GAIN:.1%}) '
      f'→ {"PASS" if checks["i_wa_telex_real_ge_5pp"] else "FAIL"}')
print(f'  (ii)  Δover-correction test = {DELTA_OC_TEST:+.2%} (<= +{VERDICT_MAX_OC_DELTA_TEST:.1%}) '
      f'→ {"PASS" if checks["ii_oc_test_le_plus01pp"] else "FAIL"}')
print(f'  (iii) ΔF1 test (PP) = {DELTA_F1_TEST:+.2%} (>= {VERDICT_MIN_F1_DELTA_TEST:.1%}) '
      f'→ {"PASS" if checks["iii_f1_test_ge_minus01pp"] else "FAIL"}')
print(f'  E3    câu sạch test bị đụng = {E3_CLEAN_TOUCHED_TEST} (<= {E3_MAX_CLEAN_TOUCHED}) '
      f'→ {"PASS" if E3_OK else "ĐIỀU TRA"}')
print(f'  Slice telex test: {len(TELEX_SLICE_TEST)} câu → {_slice_note}')
if DELTA_F1_SLICE is not None:
    print(f'  (tham khảo) ΔF1 slice telex test (PP): {DELTA_F1_SLICE:+.2%}')
print()
if PASS_HUONG_A:
    print('>> PASS HƯỚNG A: input normalization telex/VNI nhận — đề xuất cập nhật REPORT §15 / DESIGN §12 / PROJECT §8 (chờ xác nhận riêng theo PROJECT.md §0).')
else:
    print('>> FAIL: negative result — telex là giới hạn khó chạm bằng cả 2 đường augmentation (nb7) lẫn normalization; ghi vào REPORT.')

# --- QA 10 mẫu ---
qa_samples = []
for _i in TELEX_SLICE_TEST:
    if len(qa_samples) >= 10:
        break
    qa_samples.append({
        'row_id': test_recs[_i].get('row_id'),
        'src': test_recs[_i]['text'],
        'src_norm': stats_impact['test']['texts_norm'][_i],
        'gold': test_recs[_i]['corrected_text'],
        'pred_raw_pp': AB['test']['raw_pp_preds'][_i],
        'pred_norm_pp': AB['test']['norm_pp_preds'][_i],
        'changed_tokens': stats_impact['test']['changed'][_i],
    })
print()
print('=== QA · test — 10 mẫu đầu bị normalizer đổi ===')
for s in qa_samples:
    print(f"[{s['row_id']}]")
    print('  SRC      :', s['src'])
    print('  SRC_NORM :', s['src_norm'])
    print('  GOLD     :', s['gold'])
    print('  RAW+PP   :', s['pred_raw_pp'])
    print('  NORM+PP  :', s['pred_norm_pp'])
    print('  tokens   :', [(c['before'], '→', c['after']) for c in s['changed_tokens']])
    print()
if not qa_samples:
    print('(Không có mẫu nào bị normalizer đổi trên test)')


# --- Xuất file còn lại ---
def save_postproc_jsonl(recs, preds_raw_branch, preds_post, out_path):
    """Nhánh norm sau PP: prediction = sau rule, prediction_raw = trước rule (như schema nb6)."""
    with open(out_path, 'w', encoding='utf-8') as f:
        for r, p_raw, p_post in zip(recs, preds_raw_branch, preds_post):
            f.write(json.dumps({'row_id': r.get('row_id'), 'text': r['text'],
                                'corrected_text': r['corrected_text'],
                                'prediction': p_post, 'prediction_raw': p_raw},
                               ensure_ascii=False) + '\n')


for _name, _recs, _fname in [('val', val_recs, 'predictions_val_run2_norm_postproc.jsonl'),
                             ('test', test_recs, 'predictions_test_run2_norm_postproc.jsonl'),
                             ('evalreal', evalreal_recs, 'predictions_evalreal_run2_norm_postproc.jsonl')]:
    save_postproc_jsonl(_recs, AB[_name]['norm_preds'], AB[_name]['norm_pp_preds'], OUTPUT_DIR / _fname)
OUT_FILES += ['predictions_val_run2_norm_postproc.jsonl', 'predictions_test_run2_norm_postproc.jsonl',
              'predictions_evalreal_run2_norm_postproc.jsonl']


def _metrics_pack(m):
    return {'precision': m['detection']['precision'], 'recall': m['detection']['recall'],
            'f1': m['detection']['f1'], 'correction_accuracy': m['correction']['accuracy'],
            'over_correction_rate': m['over_correction']['rate'],
            'clean_retention_rate': m['over_correction']['clean_retention_rate']}


telex_ab_report = {
    'created': RUN_STAMP,
    'notebook': 'nb8_telex_normalization',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'pre_registered_thresholds': {
        'min_wa_telex_real_gain': VERDICT_MIN_WA_TELEX_GAIN,
        'max_oc_delta_test': VERDICT_MAX_OC_DELTA_TEST,
        'min_f1_delta_test': VERDICT_MIN_F1_DELTA_TEST,
        'e3_max_clean_touched': E3_MAX_CLEAN_TOUCHED,
        'min_telex_slice_n': MIN_TELEX_SLICE_N,
    },
    'config': {
        'seed': SEED,
        'adapter': 'lora_adapter_run2 (nb3) — frozen',
        'decode': 'greedy (num_beams=1)',
        'eval_batch_size': EVAL_BATCH_SIZE,
        'min_marked_tokens': MIN_MARKED_TOKENS,
        'raw_evalreal_source': RAW_EVALREAL_SOURCE,
        'evalreal_provenance': bench_meta,
        'postprocessing': {'name': PP_NAME, 'config': PP_CFG,
                           'source': 'postprocess_config.json nb6 — nạp frozen, KHÔNG re-tune'},
    },
    'run2_reference_report13': RUN2_REF,
    'ab_metrics': {_name: {_br: _metrics_pack(AB[_name][_br]) for _br in ('raw', 'raw_pp', 'norm', 'norm_pp')}
                   for _name in AB},
    'ab_pp_stats': {_name: {'raw_pp': AB[_name]['raw_pp_stats'], 'norm_pp': AB[_name]['norm_pp_stats']}
                    for _name in AB},
    'delta_verdict': {
        'wa_telex_real_norm_pp_minus_raw_pp': DELTA_WA_TELEX,
        'f1_test_norm_pp_minus_raw_pp': DELTA_F1_TEST,
        'oc_test_norm_pp_minus_raw_pp': DELTA_OC_TEST,
        'f1_slice_telex_test_pp': DELTA_F1_SLICE,
        'checks': checks, 'e3_pass': E3_OK, 'pass_huong_a': PASS_HUONG_A, 'slice_note': _slice_note,
    },
    'word_acc_evalreal': wa_evalreal,
    'qa_samples_test': qa_samples,
    'files': sorted(set(OUT_FILES + ['telex_ab_report.json'])),
}
with open(OUTPUT_DIR / 'telex_ab_report.json', 'w', encoding='utf-8') as f:
    json.dump(telex_ab_report, f, ensure_ascii=False, indent=2)

print('=== HOÀN TẤT nb8 (Hướng A) ===')
print('Đã ghi vào', OUTPUT_DIR)
for fn in telex_ab_report['files']:
    print(' -', fn)